## Chatbots with Message History using LangChain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000020D0F9D2900>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020D0F9D34D0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, my name is Vishal Goyal and here I am working as a data scientist and learning AgenticAI.")])

AIMessage(content="Hello Vishal Goyal, it's nice to meet you.  It's great that you're working as a data scientist and exploring AgenticAI. AgenticAI is a relatively new and exciting field that combines AI, cognitive architectures, and agent-based modeling. \n\nAs a data scientist, you likely have a strong foundation in machine learning, statistics, and data analysis. AgenticAI can help you take your skills to the next level by allowing you to model complex systems, simulate behaviors, and make more informed decisions.\n\nWhat specific aspects of AgenticAI are you most interested in learning about, and how do you see yourself applying it in your work as a data scientist?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 137, 'prompt_tokens': 59, 'total_tokens': 196, 'completion_time': 0.456175145, 'prompt_time': 0.013689987, 'queue_time': 0.19662396, 'total_time': 0.469865132}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, my name is Vishal Goyal and here I am working as a data scientist and learning AgenticAI."),
        AIMessage(content="Nice to meet you, Vishal Goyal. It's great to hear that you're working as a data scientist and exploring AgenticAI. AgenticAI is a fascinating field that combines artificial intelligence, machine learning, and autonomous systems. What specific aspects of AgenticAI are you interested in or currently learning about? Are you working on any projects or applications that involve AgenticAI? I'm here to help and provide any guidance or support you might need."),
        HumanMessage(content="Hey, what's my name, what I do and learning ?")
    ]
)

AIMessage(content="Your name is Vishal Goyal, you work as a Data Scientist, and you're learning AgenticAI.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 175, 'total_tokens': 199, 'completion_time': 0.05971936, 'prompt_time': 0.050549779, 'queue_time': 0.17660649, 'total_time': 0.110269139}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--a1467f7d-8c2f-490d-99b7-306b0d9de557-0', usage_metadata={'input_tokens': 175, 'output_tokens': 24, 'total_tokens': 199})

### Message History
We can use a Message History class to wrap our model and make it stateful.

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [6]:
store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history1=RunnableWithMessageHistory(model,get_session_history)

In [7]:
config1={"configurable":{"session_id":"chat1"}}

In [8]:
response=with_message_history1.invoke(
    [HumanMessage(content="Hi, my name is Vishal Goyal and here I am working as a data scientist and learning AgenticAI.")],
    config=config1
)

In [9]:
response.content

"Hi Vishal Goyal, it's nice to meet you. As a data scientist, you must be familiar with the latest advancements in AI and machine learning. AgenticAI is an exciting field that focuses on creating autonomous agents that can learn, adapt, and interact with their environment. \n\nWhat specific aspects of AgenticAI are you currently exploring, and how do you think it can be applied to your work as a data scientist? I'm here to help and provide any guidance or information you might need."

In [10]:
with_message_history1.invoke(
    [HumanMessage(content="What's my name, and what I Do?")],
    config=config1,
)

AIMessage(content='Your name is Vishal Goyal, and you work as a data scientist, currently learning about AgenticAI.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 181, 'total_tokens': 205, 'completion_time': 0.071203537, 'prompt_time': 0.117504806, 'queue_time': 0.330794533, 'total_time': 0.188708343}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--7e058e7e-c37d-4cfc-9377-d82f9aaa5e58-0', usage_metadata={'input_tokens': 181, 'output_tokens': 24, 'total_tokens': 205})

In [11]:
config2={"configurable":{"session_id":"chat2"}}

response=with_message_history1.invoke(
    [HumanMessage(content="Whats my name")],
    config=config2
)
response.content

"I don't know your name. I'm a large language model, I don't have the ability to recall personal information about individuals, and our conversation just started, so I haven't had a chance to learn your name. If you'd like to share your name, I'd be happy to chat with you."

In [12]:
response=with_message_history1.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is Vishal Goyal.'

In [13]:
response=with_message_history1.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config2
)
response.content

"Nice to meet you, John. It's great to have a name to associate with our conversation. How's your day going so far, John? Is there something I can help you with or would you like to chat about something in particular?"

In [14]:
response=with_message_history1.invoke(
    [HumanMessage(content="Whats my name")],
    config=config2
)
response.content

'I remember, your name is John. We just established that a moment ago.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. 
In this case, the raw user input is just a message, which we are passing to the LLM.

In [15]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

In [16]:
chain=prompt|model

In [17]:
chain.invoke({"messages": [HumanMessage(content="Hi, My name is Vishal Goyal and I am learning Agentic AI.")]})

AIMessage(content="Hello Vishal Goyal, nice to meet you. Agentic AI is a fascinating field that combines artificial intelligence, cognitive architectures, and software agents to create autonomous systems that can perceive, reason, and act. What specific aspects of Agentic AI are you currently learning or interested in exploring? I'd be happy to help you with any questions or provide more information on the topic.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 68, 'total_tokens': 145, 'completion_time': 0.222991084, 'prompt_time': 0.013086027, 'queue_time': 0.058507326, 'total_time': 0.236077111}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--47cffd64-6530-4c74-92b8-e2af9da0018f-0', usage_metadata={'input_tokens': 68, 'output_tokens': 77, 'total_tokens': 145})

In [18]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [19]:
config3 = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Vishal Goyal and I am learning Agentic AI.")],
    config=config3
)

response.content

"Nice to meet you, Vishal Goyal. Agentic AI is a fascinating field that combines artificial intelligence, cognitive architectures, and agent-based modeling to create autonomous systems that can perceive, reason, and act in complex environments.\n\nWhat specific aspects of Agentic AI are you interested in learning about? Are you looking to develop your skills in areas like agent-based modeling, decision-making, or multi-agent systems? Or do you have a particular application in mind, such as robotics, finance, or healthcare? I'm here to help and provide guidance to the best of my abilities."

In [20]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name and what I Do?")],
    config=config3,
)

response.content

'Your name is Vishal Goyal, and you are learning Agentic AI.'

## Add more complexity

In [ ]:
new_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

new_chain = new_prompt | model

In [30]:
new_response= new_chain.invoke(
    {
        "messages": [HumanMessage(content="Hi, My name is Vishal Goyal and I am learning Agentic AI.")],

        "language":"Hindi"})

new_response.content

'नमस्ते विशाल गोयल जी, \nमैं आपकी सहायता के लिए यहाँ हूँ। एजेंटिक एआई एक दिलचस्प विषय है, और मैं आपको इस विषय में जानने में मदद करने के लिए तैयार हूँ। एजेंटिक एआई में आपको किस तरह की जानकारी चाहिए? क्या आप इसके मूल概念, इसके अनुप्रयोग, या इसके भविष्य के बारे में जानना चाहते हैं?'

## Let's now wrap this more complicated chain in a Message History class.
### This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [33]:
new_with_message_history=RunnableWithMessageHistory(
    new_chain,
    get_session_history,
    input_messages_key="messages"
)

In [35]:
config4 = {"configurable": {"session_id": "chat4"}}

new_repsonse1=new_with_message_history.invoke(
    {
        "messages": [HumanMessage(content="Hi, My name is Vishal Goyal and I am learning Agentic AI.")],
        "language":"Hindi"
    },
    config=config4
)
new_repsonse1.content

'नमस्ते विशाल जी, एजेंटिक एआई एक बहुत ही रोचक और गतिशील क्षेत्र है, और मैं आपको इसमें सीखने में मदद करने के लिए यहाँ हूँ। एजेंटिक एआई में एजेंट्स की भूमिका और उनके द्वारा किए जाने वाले निर्णयों का अध्ययन किया जाता है, जो कि कृत्रिम बुद्धिमत्ता के क्षेत्र में एक महत्वपूर्ण योगदान करता है।\n\nआपको एजेंटिक एआई में क्या सबसे ज्यादा आकर्षित करता है? क्या आप इसके अनुप्रयोगों में रुचि रखते हैं, जैसे कि:\n\n* रोबोटिक्स: जहां एजेंट्स शारीरिक वातावरण में काम करते हैं\n* गेम théорी: जहां एजेंट्स दूसरे एजेंट्स के साथ बातचीत करते हैं\n* मशीन लर्निंग: जहां एजेंट्स डेटा से सीखते हैं और निर्णय लेते हैं\n* या फिर इसके सैद्धांतिक पहलुओं में: जहां एजेंट्स के निर्णय लेने की प्रक्रिया का अध्ययन किया जाता है\n\nमैं आपके प्रश्नों का उत्तर देने और आपको इस विषय में गहराई से समझने में मदद करने के लिए यहाँ हूँ।'

In [36]:
new_response2 = new_with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config4,
)

In [37]:
new_response2.content

'आपका नाम विशाल गोयल है।'

## Managing the Conversation History

One important concept to understand when building chatbots is how to manage conversation history. 

If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. 

## Therefore, it is important to add a step that limits the size of the messages you are passing in.

'trim_messages' helper to reduce how many messages we're sending to the model. 

The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages...

In [38]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

In [39]:
messages = [
    SystemMessage(content="you're a good assistant"),
    
    HumanMessage(content="hi! I'm Vishal Goyal."),
    AIMessage(content="hi!"),

    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

d:\My_Work\UdeMy\MyAgenticAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [42]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

new_chain=(
    RunnablePassthrough.assign(messages = itemgetter("messages") | trimmer )
    | new_prompt
    | model
)

In [44]:
new_response=new_chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What ice cream do i like")],
        "language":"Hindi"
    }
)
new_response.content

'मुझे खेद है, लेकिन मैं आपको नहीं जानता, इसलिए मैं आपकी पसंदीदा आइस्क्रीम के बारे में नहीं बता सकता। क्या आप मुझे बताना चाहेंगे?'

In [46]:
response = new_chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "Hindi",
    }
)
response.content

'आपने 2 + 2 का मैथ प्रॉब्लम पूछा था।'

### Lets wrap this in the Message History

In [ ]:
with_message_history = RunnableWithMessageHistory(
    new_chain,
    get_session_history,
    input_messages_key="messages",
)

config5 = {"configurable":{"session_id":"chat5"}}

In [49]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="what's my name?")],
        "language": "Hindi",
    },
    config=config5,
)

response.content

'मुझे खेद है, लेकिन मुझे आपका नाम नहीं पता है। हमारी बातचीत अभी शुरू हुई है और आपने मुझे अपना नाम नहीं बताया है।'

In [50]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "Hindi",
    },
    config=config5,
)

response.content

'आपके द्वारा कोई गणित का प्रश्न नहीं पूछा गया था। आप गणित का कोई प्रश्न पूछना चाहते हैं तो मैं आपकी सहायता करने के लिए तैयार हूँ।'